# Predictor de Tráfico

Este cuaderno recoge el flujo completo del proyecto: carga de datos desde la API, preparación del conjunto de entrenamiento, entrenamiento del modelo y la interfaz de predicción.

La idea es mantener en un solo sitio el código y las explicaciones que estaban repartidas en comentarios dentro de los scripts.

## Carga de datos

Primero accedemos al paquete de datos publicado en la API de Tenerife, localizamos el recurso en formato TXT y lo leemos como un DataFrame de pandas.

In [ ]:
import requests
import pandas as pd

def cargar_datos_api(url, id_paquete):
    endpoint = f"{url}/api/3/action/package_show"
    respuesta = requests.get(endpoint, params={"id": id_paquete}, timeout=20)

    if respuesta.status_code != 200:
        try:
            detalle_error = respuesta.json().get("error", {})
        except ValueError:
            detalle_error = respuesta.text
        raise ConnectionError(
            f"Error en la petición. Código: {respuesta.status_code}. Detalle: {detalle_error}"
        )

    datos_json = respuesta.json()
    recursos = datos_json.get('result', {}).get('resources', [])
    url_descarga = None

    for recurso in recursos:
        if recurso.get('format', '').upper() == 'TXT':
            url_descarga = recurso.get('url')
            break

    if not url_descarga:
        raise ValueError("No se encontró ningún fichero de datos TXT")

    df = pd.read_csv(url_descarga)
    return df

## Preprocesamiento

En esta etapa formateamos la fecha, unificamos el tráfico ligero y pesado, calculamos el sentido del carril y agrupamos los vehículos por fecha, tramo y sentido.

Después extraemos variables temporales para que el modelo pueda aprender patrones por hora y día de la semana, y codificamos la estación para trabajar con valores numéricos.

In [ ]:
from sklearn.preprocessing import LabelEncoder

def preprocesar_datos(df):
    df['fecha'] = pd.to_datetime(df['fecha'])
    df['total_vehiculos'] = df['ligeros'] + df['pesados']

    df['sentido'] = df['carril_id'].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))

    df_agrupado = df.groupby(['fecha', 'estacion_nombre', 'sentido'], as_index=False)['total_vehiculos'].sum()

    df_agrupado['hora'] = df_agrupado['fecha'].dt.hour
    df_agrupado['dia_semana'] = df_agrupado['fecha'].dt.dayofweek

    le_estacion = LabelEncoder()
    df_agrupado['estacion_encoded'] = le_estacion.fit_transform(df_agrupado['estacion_nombre'])

    features = ['estacion_encoded', 'sentido', 'hora', 'dia_semana']
    X = df_agrupado[features]
    y = df_agrupado['total_vehiculos']

    return X, y, le_estacion, df_agrupado

def preparar_entrada_usuario(estacion, sentido, fecha_str, le_estacion):
    fecha = pd.to_datetime(fecha_str)
    hora = fecha.hour
    dia_semana = fecha.dayofweek
    estacion_encoded = le_estacion.transform([estacion])[0]

    entrada = pd.DataFrame({
        'estacion_encoded': [estacion_encoded],
        'sentido': [sentido],
        'hora': [hora],
        'dia_semana': [dia_semana]
    })

    return entrada

## Entrenamiento del modelo

Dividimos los datos en entrenamiento y prueba, ajustamos un Random Forest y medimos el error con MAE. Esa métrica resume cuántos vehículos nos equivocamos, de media, en la predicción.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def entrenar_modelo(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    modelo = RandomForestRegressor(n_estimators=100, random_state=42)
    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, predicciones)

    print(f"Error Absoluto Medio (MAE): {mae:.2f} vehículos")
    return modelo

## Interfaz de predicción

La aplicación final usa Streamlit para cargar los datos una sola vez, entrenar el modelo con caché y ofrecer un formulario donde elegir tramo, sentido, fecha y hora.

Cuando el usuario pulsa predecir, se construye la entrada con las mismas variables del entrenamiento y se devuelve el volumen estimado de vehículos por hora.

In [ ]:
import pandas as pd
import streamlit as st

URL_BASE = "https://datos.tenerife.es/ckan"
ID_PAQUETE = "d02ccf54-aca2-4a82-9ef0-fbd285b5f401"

st.set_page_config(page_title="Predictor de Tráfico", layout="centered")

st.title("Predictor de Tráfico")
st.write("Selecciona un tramo, un sentido y una fecha para estimar el volumen de vehículos.")

@st.cache_data(show_spinner="Descargando y procesando datos...")
def cargar_datos_preparados():
    dataset_crudo = cargar_datos_api(URL_BASE, ID_PAQUETE)
    return preprocesar_datos(dataset_crudo)

@st.cache_resource(show_spinner="Entrenando modelo...")
def obtener_modelo(X, y):
    return entrenar_modelo(X, y)

try:
    X, y, le_estacion, df_agrupado = cargar_datos_preparados()
    modelo_entrenado = obtener_modelo(X, y)

    col1, col2 = st.columns(2)
    col1.metric("Registros", f"{len(df_agrupado):,}")
    col2.metric("Tramos", f"{df_agrupado['estacion_nombre'].nunique():,}")

    with st.form("form_prediccion"):
        estaciones = sorted(df_agrupado["estacion_nombre"].unique())
        estacion = st.selectbox("Tramo", estaciones)

        sentidos_disponibles = sorted(
            df_agrupado[df_agrupado["estacion_nombre"] == estacion]["sentido"].unique()
        )
        sentido = st.selectbox(
            "Sentido",
            sentidos_disponibles,
            format_func=lambda valor: {
                1: "Creciente (Ej: Hacia Santa Cruz)",
                -1: "Decreciente (Ej: Hacia el Sur)",
                0: "Único sentido (Ej: Calle de un solo sentido)",
            }.get(valor, str(valor)),
        )

        fecha = st.date_input("Fecha")
        hora = st.time_input("Hora")

        enviar = st.form_submit_button("Predecir")

    if enviar:
        fecha_hora = pd.Timestamp.combine(fecha, hora)
        fecha_str = fecha_hora.strftime("%Y-%m-%d %H:%M")

        entrada = preparar_entrada_usuario(estacion, sentido, fecha_str, le_estacion)
        prediccion = modelo_entrenado.predict(entrada)[0]

        texto_sentido = "Creciente" if sentido == 1 else ("Decreciente" if sentido == -1 else "Único")

        st.success("Predicción generada correctamente")
        st.subheader("Resultado")
        st.write(f"**Tramo:** {estacion}")
        st.write(f"**Sentido:** {texto_sentido}")
        st.write(f"**Fecha:** {fecha_str}")
        st.metric("Volumen estimado", f"{prediccion:.0f} vehículos/hora")

except Exception as e:
    st.error(f"No se pudo cargar el panel: {e}")

## Resumen

Este cuaderno deja documentado el proyecto completo. Si quieres, el siguiente paso puede ser separar una versión más didáctica, con celdas ejecutables paso a paso y ejemplos de salida intermedios.